<a href="https://colab.research.google.com/github/Vaishnavi639/GenAI/blob/main/22610081.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch torch-pruning psutil


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.3 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn.utils.prune as prune
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import time
import psutil
import os


In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def measure_performance(model, inputs, label="Model"):
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024 / 1024

    model.eval()
    with torch.no_grad():
        start_time = time.time()
        for _ in range(100):
            outputs = model(**inputs)
        end_time = time.time()

    inference_time = (end_time - start_time) / 100 * 1000
    mem_after = process.memory_info().rss / 1024 / 1024
    model_size = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024 / 1024

    print(f"\n {label}")
    print(f"model size: {model_size:.2f} MB")
    print(f"memory usage: {mem_after:.2f} MB")
    print(f"inference time: {inference_time:.4f} ms")

    return model_size, mem_after, inference_time

In [ ]:
measure_performance(model, inputs, "original model")

pruned_model = AutoModelForSequenceClassification.from_pretrained(model_name)
for name, module in pruned_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.3)
        prune.remove(module, 'weight')

measure_performance(pruned_model, inputs, "pruned model")



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



- original model -
Model Size: 255.41 MB
Memory Usage: 1630.93 MB
Inference Time: 53.2630 ms

- Pruned Model (30%) -
Model Size: 255.41 MB
Memory Usage: 2096.16 MB
Inference Time: 43.5876 ms


(255.41309356689453, 2096.15625, 43.58762741088867)

In [ ]:
quantized_model = AutoModelForSequenceClassification.from_pretrained(model_name)

quantized_model = torch.quantization.quantize_dynamic(
    quantized_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

measure_performance(quantized_model, inputs, "quantized model")




Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2195439902.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see


- Quantized Model (INT8) -
Model Size: 91.00 MB
Memory Usage: 2346.94 MB
Inference Time: 17.7448 ms


/tmp/ipython-input-2195439902.py:21: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  combined_model = torch.quantization.quantize_dynamic(



- pruned and quantized model -
Model Size: 91.00 MB
Memory Usage: 2830.73 MB
Inference Time: 33.2169 ms

--- Output Comparison ---
Original Output: tensor([-0.0164,  0.0421])
Optimized Output: tensor([-0.0187,  0.1224])
Max Difference: 0.080306


In [ ]:
combined_model = AutoModelForSequenceClassification.from_pretrained(model_name)

for name, module in combined_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.3)
        prune.remove(module, 'weight')

combined_model = torch.quantization.quantize_dynamic(
    combined_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

measure_performance(combined_model, inputs, "pruned and quantized model")

with torch.no_grad():
    original_output = model(**inputs).logits
    optimized_output = combined_model(**inputs).logits



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1793704197.py:8: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see


- pruned and quantized model -
model size: 91.00 MB
memory usage: 3042.88 MB
inference time: 16.6238 ms

 output comparison
original output: tensor([-0.0164,  0.0421])
optimized output: tensor([-0.0789, -0.0936])
max difference: 0.135729
